# Notebook 02 — Diseño del detector

Este notebook implementa las cuatro capas del detector de cierre de válvulas: sanidad (Capa 0), suavizado (Capa 1), regresión P-Q (Capa 2) y CUSUM (Capa 3). Las funciones se prototipan inline aquí y se refactorizan a `src/` una vez que se estabilizan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

## 1. Carga del dataset

In [ ]:
DATA_PATH = Path("..") / "data" / "PLOT_TS_P_Q.csv"

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    decimal=".",
    parse_dates=["TS"],
    date_format="%d/%m/%Y %H:%M:%S",
)

print(df.shape)
df.head(3)

## 2. Capa 0 — Sanidad

La regla descarta las filas donde `Q == 0` y `P > 50`. Estos cuatro registros del 12-abr (00:07–00:10) son glitches físicamente imposibles identificados en el notebook 01: si el gasoducto está presurizado (P ≈ 75), no puede haber caudal nulo.

La asimetría de la condición es deliberada: `Q = 0` con `P` alta es imposible físicamente y siempre indica error de telemetría. `Q = 0` con `P` baja, en cambio, podría corresponder a planta parada (estado legítimo) y no debe filtrarse. El umbral de 50 está bien por debajo del mínimo operativo observado (~65), con margen suficiente para no capturar ningún estado real de baja presión.

In [ ]:
def aplicar_sanidad(df: pd.DataFrame, q_max: float = 0.0,
                    p_min: float = 50.0) -> pd.DataFrame:
    """Descarta filas con Q <= q_max y P > p_min.

    Elimina glitches físicamente imposibles: caudal nulo con sistema
    presurizado. Los defaults (q_max=0.0, p_min=50.0) corresponden al
    caso validado en el notebook 01 sobre este dataset.
    Devuelve un DataFrame nuevo con índice reseteado.
    """
    mascara_glitch = (df["Q"] <= q_max) & (df["P"] > p_min)
    return df[~mascara_glitch].reset_index(drop=True)

In [ ]:
df_sano = aplicar_sanidad(df)

print(f"df      : {df.shape}")
print(f"df_sano : {df_sano.shape}")

## 2.1 Validación de la Capa 0

Tres verificaciones automáticas:
1. Que se filtraron exactamente 4 filas.
2. Que las filas filtradas corresponden a los timestamps del 12-abr identificados en el notebook 01.
3. Que la condición `Q == 0` y `P > 50` ya no aparece en `df_sano`.

In [ ]:
# 1. Cantidad de filas filtradas
n_filtradas = df.shape[0] - df_sano.shape[0]
assert n_filtradas == 4, f"Se esperaban 4 filas filtradas, se obtuvieron {n_filtradas}"
print(f"Filas filtradas: {n_filtradas}")

# 2. Timestamps de las filas filtradas
mask_filtradas = ~df["TS"].isin(df_sano["TS"])
ts_filtradas = sorted(df.loc[mask_filtradas, "TS"].tolist())
ts_esperados = [
    pd.Timestamp("2026-04-12 00:07:00"),
    pd.Timestamp("2026-04-12 00:08:00"),
    pd.Timestamp("2026-04-12 00:09:00"),
    pd.Timestamp("2026-04-12 00:10:00"),
]
assert ts_filtradas == ts_esperados, f"Timestamps incorrectos: {ts_filtradas}"
print(f"Timestamps filtrados: {[str(ts) for ts in ts_filtradas]}")

# 3. Ausencia de la condicion en la salida
n_residual = ((df_sano["Q"] == 0) & (df_sano["P"] > 50)).sum()
assert n_residual == 0, f"Quedan {n_residual} filas con Q=0 y P>50 en df_sano"
print(f"Filas con Q=0 y P>50 en df_sano: {n_residual}")

print("\u2713 Capa 0 validada")

## 3. Capa 1 — Suavizado

Se usa **rolling mean trailing** (ventana que mira hacia atrás): en cada instante `t` el valor suavizado es el promedio de las últimas N observaciones, sin incluir puntos futuros. Esto preserva la causalidad y permite usar la función tal cual en modo streaming, sin modificación.

La ventana mínima razonable es **2 horas** por dos motivos derivados del análisis del notebook 01: (1) cubre al menos un ciclo completo del heartbeat horario de P, eliminando el pulso periódico que generaría falsas alarmas en la divergencia; (2) promedia decenas de saltos de deadband (P cambia en ~3.7% de las filas, lo que da ~4–5 saltos por hora en régimen denso).

Antes de fijar la ventana definitiva se comparan visualmente **2h, 4h y 8h** sobre el tramo 18–22 abr, que corresponde al régimen denso de alta presión visible en el notebook 01.

In [ ]:
def aplicar_suavizado(df: pd.DataFrame,
                      ventana: str = "2h") -> pd.DataFrame:
    """Suaviza P y Q con rolling mean trailing de ventana temporal indicada.

    La ventana se interpreta como string de pandas (ej '2h', '4h', '30min').
    Requiere df ordenado por TS. Devuelve un DataFrame nuevo con columnas
    TS, P, Q, P_suave, Q_suave. Los primeros puntos de cada serie suavizada
    son NaN hasta que la ventana se llena.
    """
    tmp = df.set_index("TS").copy()
    rolling = tmp[["P", "Q"]].rolling(window=ventana)
    tmp["P_suave"] = rolling["P"].mean()
    tmp["Q_suave"] = rolling["Q"].mean()
    return tmp.reset_index()

In [ ]:
df_2h = aplicar_suavizado(df_sano, ventana="2h")
df_4h = aplicar_suavizado(df_sano, ventana="4h")
df_8h = aplicar_suavizado(df_sano, ventana="8h")

print(f"NaN en P_suave — 2h: {df_2h['P_suave'].isna().sum()}")
print(f"NaN en P_suave — 4h: {df_4h['P_suave'].isna().sum()}")
print(f"NaN en P_suave — 8h: {df_8h['P_suave'].isna().sum()}")

## 3.1 Comparación visual de ventanas

Se grafica el tramo 18–22 abr: cinco días de régimen denso con variabilidad típica del sistema. La serie cruda aparece en gris claro al fondo; las tres versiones suavizadas se superponen en colores distintos. El objetivo es elegir la ventana mínima que aplaste la textura de escalones del deadband sin deformar la dinámica real (tendencias de horas a días).

In [ ]:
ts_inicio = pd.Timestamp("2026-04-18 00:00")
ts_fin    = pd.Timestamp("2026-04-22 23:59")

def filtrar_tramo(d):
    return d[(d["TS"] >= ts_inicio) & (d["TS"] <= ts_fin)]

tramo_sano = filtrar_tramo(df_sano)
tramo_2h   = filtrar_tramo(df_2h)
tramo_4h   = filtrar_tramo(df_4h)
tramo_8h   = filtrar_tramo(df_8h)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for ax, col, ylabel, titulo in [
    (axes[0], "P", "P (presión)", "P — comparación de ventanas"),
    (axes[1], "Q", "Q (caudal)",  "Q — comparación de ventanas"),
]:
    ax.plot(tramo_sano["TS"], tramo_sano[col],
            color="gray", alpha=0.4, linewidth=0.6, label="Crudo")
    ax.plot(tramo_2h["TS"], tramo_2h[f"{col}_suave"],
            color="tab:blue",   linewidth=1.2, label="2h")
    ax.plot(tramo_4h["TS"], tramo_4h[f"{col}_suave"],
            color="tab:orange", linewidth=1.2, label="4h")
    ax.plot(tramo_8h["TS"], tramo_8h[f"{col}_suave"],
            color="tab:red",    linewidth=1.2, label="8h")
    ax.set_title(titulo)
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)

axes[1].set_xlabel("Tiempo")
plt.tight_layout()
plt.show()

## 3.2 Decisión de ventana

Decisión pendiente: tras inspeccionar el gráfico, elegir la ventana definitiva. Criterio: la ventana mínima que aplaste la textura de escalones del deadband sin deformar la dinámica real visible en el tramo.

## 4. Capa 2 — Regresión acotada Q=f(P) y residual

Esta capa modela la relación lineal entre P y Q usando los datos suavizados de operación, y para cada punto del dataset calcula el residual (Q_suave observado menos Q_suave predicho por la regresión).

**Por qué el residual:** la firma física del cierre aguas abajo (P sube, Q baja) rompe la relación normal Q=f(P), y el residual negativo captura ese desacople direccionalmente.

**Por qué acotada al régimen denso (P_suave > 87):** una regresión global preliminar mostró dispersión de residual inaceptable (std ≈ 77, mínimos de -317) causada por dos regímenes de operación atípica presentes en el dataset. Acotar la regresión al régimen denso especializa el modelo en el régimen operativo principal y declara las zonas con P_suave ≤ 87 como **no monitoreables** — el detector no opina fuera de su régimen entrenado.

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
UMBRAL_DENSO = 89

df_sin_nan = df_4h.dropna(subset=["P_suave", "Q_suave"])
df_train   = df_sin_nan[df_sin_nan["P_suave"] > UMBRAL_DENSO].copy()

print(f"df_train.shape           : {df_train.shape}")
print(f"Fracción del total sin NaN : {len(df_train) / len(df_sin_nan):.1%}")

In [ ]:
X = df_train[["P_suave"]].values
y = df_train["Q_suave"].values

modelo = LinearRegression().fit(X, y)

pendiente  = modelo.coef_[0]
intercepto = modelo.intercept_

print(f"Pendiente  : {pendiente:.4f}")
print(f"Intercepto : {intercepto:.4f}")
print(f"R²         : {modelo.score(X, y):.4f}")

In [ ]:
df_residual = df_4h.copy()
mask_nonan = df_residual["P_suave"].notna()

df_residual["Q_predicho"] = np.nan
df_residual.loc[mask_nonan, "Q_predicho"] = modelo.predict(
    df_residual.loc[mask_nonan, ["P_suave"]].values
)

df_residual["residual"]     = df_residual["Q_suave"] - df_residual["Q_predicho"]
df_residual["monitoreable"] = df_residual["P_suave"] > UMBRAL_DENSO

print(df_residual[df_residual["monitoreable"]]["residual"].describe())

## 4.1 Visualización del ajuste

Se grafica el scatter P_suave vs Q_suave diferenciando el régimen denso (usado en el entrenamiento, azul oscuro) de la zona no monitoreable (gris claro), con la recta de regresión superpuesta para validar visualmente el ajuste.

In [ ]:
df_plot = df_4h.dropna(subset=["P_suave", "Q_suave"])
denso    = df_plot[df_plot["P_suave"] >  UMBRAL_DENSO]
no_denso = df_plot[df_plot["P_suave"] <= UMBRAL_DENSO]

x_line = np.linspace(UMBRAL_DENSO, df_train["P_suave"].max(), 100)
y_line = modelo.predict(x_line.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(no_denso["P_suave"], no_denso["Q_suave"],
           color="lightgray", alpha=0.3, s=2,
           label=f"No monitoreable (P ≤ {UMBRAL_DENSO})")
ax.scatter(denso["P_suave"], denso["Q_suave"],
           color="darkblue", alpha=0.2, s=2, label="Régimen denso (entrenamiento)")
ax.plot(x_line, y_line, color="red", linewidth=2,
        label=f"Regresión: Q = {pendiente:.2f}·P + {intercepto:.2f}")
ax.axvline(UMBRAL_DENSO, color="black", linestyle="--", alpha=0.5,
           label=f"Umbral monitoreabilidad (P={UMBRAL_DENSO})")
ax.set_xlabel("P_suave")
ax.set_ylabel("Q_suave")
ax.set_title("Ajuste lineal Q_suave = f(P_suave) — régimen denso")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 4.2 Serie temporal del residual

Se grafica el residual a lo largo del mes, sombreando en gris las zonas no monitoreables (P_suave ≤ 87) para distinguir visualmente dónde el detector opera y dónde no.

In [ ]:
y_min = df_residual["residual"].min()
y_max = df_residual["residual"].max()

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df_residual["TS"], df_residual["residual"], color="tab:blue", linewidth=0.7)
ax.axhline(0, color="black", linestyle="--", alpha=0.5)
ax.fill_between(
    df_residual["TS"], y_min, y_max,
    where=~df_residual["monitoreable"],
    color="lightgray", alpha=0.4, label="No monitoreable",
)
ax.set_xlabel("Tiempo")
ax.set_ylabel("Residual (Q observado - Q predicho)")
ax.set_title("Residual de la regresión acotada — mes completo")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 4.3 Distribución del residual en zona monitoreable

El histograma solo incluye la zona monitoreable (P_suave > 87) porque es ahí donde el detector va a operar. Muestra si el residual está centrado en 0 y cuál es su dispersión típica.

In [ ]:
residual_monitoreable = df_residual[df_residual["monitoreable"]]["residual"].dropna()

title = f"Distribución del residual en zona monitoreable (std={residual_monitoreable.std():.2f})"

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(residual_monitoreable, bins=80, color="tab:blue", edgecolor="black", alpha=0.7)
ax.axvline(0, color="red", linestyle="--", alpha=0.7)
ax.set_xlabel("Residual")
ax.set_ylabel("Frecuencia")
ax.set_title(title)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4.4 Conclusiones

El modelo lineal acotado al régimen denso (P_suave > 89) modela adecuadamente la relación direccional Q = f(P) en el corazón del régimen operativo principal del sistema. R² = 0.20 y std del residual = 46 unidades de Q. El 54% del dataset queda dentro de la zona monitoreable.

El R² bajo refleja una característica estructural del sistema: la variabilidad de Q dentro del régimen denso responde a factores adicionales no presentes en el dataset (demanda downstream, temperatura, ciclos diarios, etc.). Este techo no se resuelve con más calibración del umbral; es información intrínseca faltante.

La decisión de avanzar a Capa 3 con este modelo se sostiene en que la firma física del cierre es **direccional** (residual negativo sostenido) y la detección por CUSUM no requiere un ajuste fino del modelo subyacente, sino capacidad de identificar sesgos persistentes en el residual. La calidad del modelo se validará empíricamente en el notebook 03 mediante fault injection.

**Parámetros del modelo final:** pendiente = 15.09, intercepto = -562.86, umbral de monitoreabilidad P_suave > 89.